# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the published FAIRˆ<sup>2</sup> dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata (no subscripting of .metadata)
dataset = mlc.Dataset(croissant_url)
metadata_dict = json.loads(dataset.metadata.json())  # For display only

print("Dataset Name:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("Version:", dataset.metadata.version)
print("Published:", dataset.metadata.datePublished)


## 2. Data Overview
Review available record sets, their `@id`s, and overview fields. All entities are referenced via their Croissant `@id` for reproducibility and clarity.

In [ ]:
# List available record sets and display their field IDs
from pprint import pprint

record_set_objs = list(dataset.metadata.recordSets)

print(f"Found {len(record_set_objs)} record sets:")
for rset in record_set_objs:
    print(f"RecordSet @id: {rset.id}")
    print(f"  name       : {rset.name}")
    print(f"  description: {getattr(rset, 'description', '')}")
    print("  Fields (by @id):")
    for field in rset.fields:
        print(f"    - {field.id} (name: {field.name}, type: {field.dataType})")
    print()


## 3. Data Extraction
Load data from each record set into a DataFrame using the record set and field `@id`s found above. Each DataFrame key is the record set `@id`.

In [ ]:
# Extract all data as DataFrames, indexed by record set @id.
dataframes = {}
record_set_ids = [rs.id for rs in record_set_objs]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

chosen_record_set_id = record_set_ids[0]
print(f"Columns in DataFrame for record set {chosen_record_set_id}:")
print(dataframes[chosen_record_set_id].columns.tolist())
dataframes[chosen_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, and grouping, using only field `@id`s.

- **Step 1:** Choose a numeric field `@id` to filter and normalize (replace with your field of interest found in section 2).
- **Step 2:** Demonstrate filtering and normalization, then group by a field if appropriate.


In [ ]:
# Example: Use first record set and select numeric field/column by @id.
df = dataframes[chosen_record_set_id]

# Inspect all numeric columns and their @ids
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric fields in {chosen_record_set_id}:", numeric_fields)

# Choose a numeric field for filtering (replace with the most relevant one for your analysis)
numeric_field_id = numeric_fields[0] if numeric_fields else None

if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records in '{chosen_record_set_id}' with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} rows")
    
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (filtered_df[numeric_field_id].std() + 1e-8)
    print(f"Normalized field '{numeric_field_id}' (first 5 rows):")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Try grouping by another field -- pick a non-numeric column if available
    group_fields = [col for col in df.columns if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col])]
    group_field_id = group_fields[0] if group_fields else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}' (first 5 groups):")
        print(grouped_df.head())
else:
    print("No numeric fields found to run EDA.")


## 5. Visualization
Visualize the distribution and relationships of numeric fields, referencing each by the field `@id`. Adjust the fields below to match interesting variables in your dataset.

In [ ]:
# Visualize the normalized numeric_field_id if available
if numeric_field_id is not None and len(filtered_df) > 0:
    # Histogram of values
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Histogram of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If grouped, show a boxplot by group field
    if group_field_id is not None:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field for visualization.")


## 6. Conclusion
In this notebook, we demonstrated how to access and process the FAIR ˆ<sup>2</sup> dataset using the `mlcroissant` library by referencing all record sets and field entities by their Croissant `@id`. We extracted and explored the tabular content, normalized a value field, filtered out lower records, and visualized basic distributions. This workflow can be extended to domain-specific analysis and supports reproducible FAIR data processing.